# 02 · Prata (requisito 06)
Tipos corretos, inválidos barrados por regras de qualidade, uma linha por chave e **MERGE** na Prata.

| Tabela | Chave (o que identifica uma linha) |
|---|---|
| `prata_clima` | `codigo_ibge` + `data_hora` (um município numa hora) |
| `prata_ibge` | `codigo_ibge` |
| `prata_relatos` | `id_relato` |

Ordem: IBGE primeiro (o cadastro de municípios é usado na regra dos relatos), depois clima e relatos.
**Rode o notebook inteiro duas vezes**: na 2ª, os MERGEs devem mostrar `num_inserted_rows = 0`.

In [0]:
SET TIME ZONE 'America/Sao_Paulo';

key,value
spark.sql.session.timeZone,America/Sao_Paulo


## 1. IBGE: população por município

In [0]:
-- O Auto Loader (sem inferColumnTypes) guarda os objetos aninhados como TEXTO JSON:
-- from_json transforma esse texto de volta em estrutura antes de extrair os campos.
CREATE OR REPLACE TEMP VIEW ibge_tipado AS
SELECT CAST(loc.id AS INT)                                      AS codigo_ibge,
       trim(regexp_replace(loc.nome, '[(][A-Z]{2}[)]$', ''))    AS municipio,  -- "São Paulo (SP)" -> "São Paulo"
       regexp_extract(loc.nome, '[(]([A-Z]{2})[)]$', 1)         AS uf,         -- "São Paulo (SP)" -> "SP"
       CAST(ser['2022'] AS BIGINT)                              AS populacao,
       2022                                                     AS ano_referencia,
       carregado_em
FROM (
  SELECT from_json(localidade, 'id STRING, nome STRING') AS loc,
         from_json(serie, 'MAP<STRING, STRING>')         AS ser,
         carregado_em
  FROM workspace.gs.bronze_ibge
);

In [0]:
-- Regras de qualidade do IBGE: quantas linhas cada uma barra
CREATE OR REPLACE TEMP VIEW regras_ibge AS
SELECT 'ibge' AS fonte, 'R1' AS regra, 'código IBGE nulo' AS motivo,
       SUM(CASE WHEN codigo_ibge IS NULL THEN 1 ELSE 0 END) AS linhas_barradas FROM ibge_tipado
UNION ALL
SELECT 'ibge', 'R2', 'população nula ou <= 0',
       SUM(CASE WHEN populacao IS NULL OR populacao <= 0 THEN 1 ELSE 0 END) FROM ibge_tipado;

CREATE OR REPLACE TEMP VIEW lote_prata_ibge AS
SELECT codigo_ibge, municipio, uf, populacao, ano_referencia
FROM ibge_tipado
WHERE codigo_ibge IS NOT NULL AND populacao > 0
QUALIFY ROW_NUMBER() OVER (PARTITION BY codigo_ibge ORDER BY carregado_em DESC) = 1;

CREATE TABLE IF NOT EXISTS workspace.gs.prata_ibge AS SELECT * FROM lote_prata_ibge LIMIT 0;

num_affected_rows,num_inserted_rows


In [0]:
MERGE INTO workspace.gs.prata_ibge AS alvo
USING lote_prata_ibge AS novo
ON alvo.codigo_ibge = novo.codigo_ibge
WHEN MATCHED AND NOT (alvo.populacao <=> novo.populacao AND alvo.municipio <=> novo.municipio) THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


## 2. Clima: uma linha por município por hora
A Open-Meteo devolve listas paralelas (`hourly.time`, `hourly.precipitation`, ...). `arrays_zip` junta as listas e
`explode` abre uma linha por horário. Sem `inferColumnTypes`, tudo chega como texto: convertemos com `CAST`.

In [0]:
CREATE OR REPLACE TEMP VIEW clima_horario AS
SELECT CAST(codigo_ibge AS INT)                AS codigo_ibge,
       municipio,
       CAST(latitude AS DOUBLE)                AS latitude,
       CAST(longitude AS DOUBLE)               AS longitude,
       to_timestamp(h.time)                    AS data_hora,
       CAST(h.precipitation AS DOUBLE)         AS chuva_mm,
       CAST(h.temperature_2m AS DOUBLE)        AS temperatura_c,
       CAST(h.relative_humidity_2m AS DOUBLE)  AS umidade_pct,
       CAST(h.wind_gusts_10m AS DOUBLE)        AS rajada_kmh,
       CAST(h.weather_code AS INT)             AS codigo_tempo_wmo,
       to_date(data_coleta)                    AS data_coleta,
       carregado_em
FROM (
  SELECT codigo_ibge, municipio, latitude, longitude, data_coleta, carregado_em,
         explode(arrays_zip(hr.time, hr.precipitation, hr.temperature_2m,
                            hr.relative_humidity_2m, hr.wind_gusts_10m, hr.weather_code)) AS h
  FROM (
    -- hourly chega como texto JSON: from_json devolve as listas paralelas como arrays
    SELECT *, from_json(hourly, 'time ARRAY<STRING>, precipitation ARRAY<STRING>, temperature_2m ARRAY<STRING>,
                                 relative_humidity_2m ARRAY<STRING>, wind_gusts_10m ARRAY<STRING>,
                                 weather_code ARRAY<STRING>') AS hr
    FROM workspace.gs.bronze_clima
  )
);

In [0]:
-- Regras de qualidade do clima
CREATE OR REPLACE TEMP VIEW regras_clima AS
SELECT 'clima' AS fonte, 'R1' AS regra, 'data/hora nula' AS motivo,
       SUM(CASE WHEN data_hora IS NULL THEN 1 ELSE 0 END) AS linhas_barradas FROM clima_horario
UNION ALL
SELECT 'clima', 'R2', 'chuva nula ou negativa',
       SUM(CASE WHEN chuva_mm IS NULL OR chuva_mm < 0 THEN 1 ELSE 0 END) FROM clima_horario
UNION ALL
SELECT 'clima', 'R3', 'temperatura nula ou fora de -10 a 50 °C',
       SUM(CASE WHEN temperatura_c IS NULL OR temperatura_c NOT BETWEEN -10 AND 50 THEN 1 ELSE 0 END) FROM clima_horario;

-- Lote limpo, uma linha por chave (a coleta mais recente vence: previsão atualizada substitui a antiga)
CREATE OR REPLACE TEMP VIEW lote_prata_clima AS
SELECT codigo_ibge, municipio, latitude, longitude, data_hora, chuva_mm, temperatura_c,
       umidade_pct, rajada_kmh, codigo_tempo_wmo, data_coleta
FROM clima_horario
WHERE data_hora IS NOT NULL
  AND chuva_mm >= 0
  AND temperatura_c BETWEEN -10 AND 50
QUALIFY ROW_NUMBER() OVER (PARTITION BY codigo_ibge, data_hora ORDER BY carregado_em DESC) = 1;

CREATE TABLE IF NOT EXISTS workspace.gs.prata_clima AS SELECT * FROM lote_prata_clima LIMIT 0;

num_affected_rows,num_inserted_rows


In [0]:
MERGE INTO workspace.gs.prata_clima AS alvo
USING lote_prata_clima AS novo
ON alvo.codigo_ibge = novo.codigo_ibge AND alvo.data_hora = novo.data_hora
WHEN MATCHED AND NOT (alvo.chuva_mm <=> novo.chuva_mm AND alvo.temperatura_c <=> novo.temperatura_c
                      AND alvo.rajada_kmh <=> novo.rajada_kmh) THEN UPDATE SET *  -- previsão corrigida
WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


## 3. Relatos de alagamento (dados simulados pelo gerador do DAG)
O gerador injeta ~3% de registros com defeito e ~2% de reenvios duplicados, imitando falhas reais de um app.

In [0]:
CREATE OR REPLACE TEMP VIEW relatos_tipados AS
SELECT id_relato,
       to_timestamp(data_hora)          AS data_hora,
       CAST(codigo_ibge AS INT)         AS codigo_ibge,
       nome, email, telefone,
       CAST(latitude AS DOUBLE)         AS latitude,
       CAST(longitude AS DOUBLE)        AS longitude,
       lower(trim(nivel_agua))          AS nivel_agua,
       carregado_em, arquivo_origem
FROM workspace.gs.bronze_relatos;

CREATE OR REPLACE TEMP VIEW relatos_avaliados AS
SELECT r.*,
       CASE
         WHEN r.data_hora IS NULL                                   THEN 'R1 - data/hora nula'
         WHEN m.codigo_ibge IS NULL                                 THEN 'R2 - município fora do cadastro IBGE'
         WHEN NOT (r.latitude BETWEEN -24.1 AND -23.1
                   AND r.longitude BETWEEN -47.2 AND -46.1)         THEN 'R3 - coordenada fora da Grande SP'
       END AS motivo_barrado
FROM relatos_tipados r
LEFT JOIN workspace.gs.prata_ibge m ON r.codigo_ibge = m.codigo_ibge;

CREATE OR REPLACE TEMP VIEW regras_relatos AS
SELECT 'relatos' AS fonte, left(motivo, 2) AS regra, substr(motivo, 6) AS motivo, linhas_barradas
FROM (SELECT m.motivo, COUNT(a.motivo_barrado) AS linhas_barradas
      FROM (VALUES ('R1 - data/hora nula'), ('R2 - município fora do cadastro IBGE'),
                   ('R3 - coordenada fora da Grande SP')) AS m(motivo)
      LEFT JOIN relatos_avaliados a ON a.motivo_barrado = m.motivo
      GROUP BY m.motivo);

In [0]:
-- Quarentena: os registros barrados ficam guardados com o motivo (não se perde nada)
CREATE OR REPLACE TABLE workspace.gs.prata_quarentena_relatos AS
SELECT * FROM relatos_avaliados WHERE motivo_barrado IS NOT NULL;

CREATE OR REPLACE TEMP VIEW lote_prata_relatos AS
SELECT id_relato, data_hora, codigo_ibge, nome, email, telefone, latitude, longitude, nivel_agua
FROM relatos_avaliados
WHERE motivo_barrado IS NULL
QUALIFY ROW_NUMBER() OVER (PARTITION BY id_relato ORDER BY carregado_em DESC, arquivo_origem DESC) = 1;

CREATE TABLE IF NOT EXISTS workspace.gs.prata_relatos AS SELECT * FROM lote_prata_relatos LIMIT 0;

num_affected_rows,num_inserted_rows


In [0]:
MERGE INTO workspace.gs.prata_relatos AS alvo
USING lote_prata_relatos AS novo
ON alvo.id_relato = novo.id_relato
WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


## 4. Resultados para o relatório

In [0]:
-- PRINT: regras de qualidade e linhas barradas por regra
CREATE OR REPLACE TABLE workspace.gs.prata_regras_qualidade AS
SELECT *, current_timestamp() AS medido_em FROM (
  SELECT * FROM regras_ibge UNION ALL SELECT * FROM regras_clima UNION ALL SELECT * FROM regras_relatos);

SELECT fonte, regra, motivo, linhas_barradas FROM workspace.gs.prata_regras_qualidade ORDER BY fonte, regra;

fonte,regra,motivo,linhas_barradas
clima,R1,data/hora nula,0
clima,R2,chuva nula ou negativa,0
clima,R3,temperatura nula ou fora de -10 a 50 °C,0
ibge,R1,código IBGE nulo,0
ibge,R2,população nula ou <= 0,0
relatos,R1,data/hora nula,60
relatos,R2,município fora do cadastro IBGE,46
relatos,R3,coordenada fora da Grande SP,62


In [0]:
-- PRINT: duplicatas removidas e total de linhas em cada camada
SELECT 'relatos' AS tabela,
       (SELECT COUNT(*) FROM workspace.gs.bronze_relatos)                       AS linhas_bronze,
       (SELECT COUNT(*) FROM relatos_avaliados WHERE motivo_barrado IS NULL)
         - (SELECT COUNT(*) FROM lote_prata_relatos)                             AS duplicatas_removidas,
       (SELECT COUNT(*) FROM workspace.gs.prata_quarentena_relatos)             AS em_quarentena,
       (SELECT COUNT(*) FROM workspace.gs.prata_relatos)                        AS linhas_prata
UNION ALL
SELECT 'clima (linhas horárias)',
       (SELECT COUNT(*) FROM clima_horario),
       (SELECT COUNT(*) FROM clima_horario WHERE data_hora IS NOT NULL AND chuva_mm >= 0 AND temperatura_c BETWEEN -10 AND 50)
         - (SELECT COUNT(*) FROM lote_prata_clima),
       0,
       (SELECT COUNT(*) FROM workspace.gs.prata_clima)
UNION ALL
SELECT 'ibge',
       (SELECT COUNT(*) FROM workspace.gs.bronze_ibge),
       (SELECT COUNT(*) FROM ibge_tipado WHERE codigo_ibge IS NOT NULL AND populacao > 0)
         - (SELECT COUNT(*) FROM lote_prata_ibge),
       0,
       (SELECT COUNT(*) FROM workspace.gs.prata_ibge);

tabela,linhas_bronze,duplicatas_removidas,em_quarentena,linhas_prata
relatos,5531,101,168,5262
clima (linhas horárias),8640,0,0,8640
ibge,360,350,0,10


In [0]:
-- PRINT: histórico dos MERGEs. Em operationMetrics, a 2ª execução mostra numTargetRowsInserted = 0
DESCRIBE HISTORY workspace.gs.prata_clima;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-11-05T18:59:38.000-03:00,76105308753725,arthurbaldisseracm@gmail.com,MERGE,"Map(predicate -> [""((codigo_ibge#30923 = codigo_ibge#30828) AND (data_hora#30927 = data_hora#30832))""], clusterBy -> [], matchedPredicates -> [{""predicate"":""((NOT (chuva_mm#30928 <=> chuva_mm#30833) OR NOT (temperatura_c#30929 <=> temperatura_c#30834)) OR NOT (rajada_kmh#30931 <=> rajada_kmh#30836))"",""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}], clusteringOnWriteStatus -> null)",null,List(75709179358508),441cfd00-ae12-4b78-8e21-66ce1c4647c8,1105-210208-szrvtpxj-v2n,1,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 0, numTargetBytesAdded -> 0, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 2703, materializeSourceTimeMs -> 507, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 989, numTargetRowsUpdated -> 0, numOutputRows -> 0, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 8640, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1160)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
1,2026-11-05T18:54:20.000-03:00,76105308753725,arthurbaldisseracm@gmail.com,MERGE,"Map(predicate -> [""((codigo_ibge#21633 = codigo_ibge#21538) AND (data_hora#21637 = data_hora#21542))""], clusterBy -> [], matchedPredicates -> [{""predicate"":""((NOT (chuva_mm#21638 <=> chuva_mm#21543) OR NOT (temperatura_c#21639 <=> temperatura_c#21544)) OR NOT (rajada_kmh#21641 <=> rajada_kmh#21546))"",""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}], clusteringOnWriteStatus -> null)",null,List(75709179358508),bcc4a446-08ab-4c1d-9abe-682cac0ae897,1105-210208-szrvtpxj-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 38662, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 2648, materializeSourceTimeMs -> 654, numTargetRowsInserted -> 8640, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 798, numTargetRowsUpdated -> 0, numOutputRows -> 8640, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 8640, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1153)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
0,2026-11-05T18:54:15.000-03:00,76105308753725,arthurbaldisseracm@gmail.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""e95d2280-772c-411b-8481-38971eda9142"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-8b104d43-3925-4ad9-af78-df0bf01251d3"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-69c81f79-895d-43e1-9f94-539420fdaf30""}, statsOnLoad -> true, clusterByAuto -> true, clusteringOnWriteStatus -> Reason for skipping: Nonexistent clustering column, generally when replacing a Liquid table with a table without clustering columns)",null,List(75709179358508),5ee7ce